# Tuldok voice maker 🎙️
Records every line the Tuldok kid app says (about 220), **with feeling**, using
[Chatterbox](https://github.com/resemble-ai/chatterbox) by Resemble AI (MIT License).

**How to use**
1. **Runtime → Change runtime type → T4 GPU → Save.**
2. Run the cells in order with ▶. Cell 1 restarts Colab once ("session crashed" is expected), then carry on with cell 2.
   After that restart, **Runtime → Run all** also works.
3. Cell 5 makes 6 test lines to listen to. Change the feeling in cell 4 until you like it.
4. Cell 6 makes all lines and downloads `tuldok-voice.zip`. With *Save to Drive* on, the clips and the zip are also kept in
   **My Drive › tuldok-voice**, so a dropped connection loses nothing: just run cells 1, 2, 4 and 6 again and it continues.
5. Put the zip into the app with the Termux command at the bottom.

In [ ]:
#@title 1. Install (restarts Colab once)
import os, shutil
from importlib.metadata import version, PackageNotFoundError

def ver(p):
    try:
        return version(p)
    except PackageNotFoundError:
        return ''

if not shutil.which('nvidia-smi'):
    print('!! No GPU yet: Runtime > Change runtime type > T4 GPU (it works without one, but very slowly).')
if ver('chatterbox-tts') and ver('torch').startswith('2.6.') and ver('torchvision').startswith('0.21.'):
    print('Already installed: chatterbox-tts', ver('chatterbox-tts'), '| torch', ver('torch'), '| torchvision', ver('torchvision'))
    print('Go on to cell 2.')
else:
    print('Installing (1-3 minutes). Ignore pip lines about starlette or other unused packages.')
    get_ipython().system('pip install -q chatterbox-tts torchvision==0.21.0')
    print('Installed. Restarting Colab now ("session crashed" is expected). Then run cell 2.')
    os.kill(os.getpid(), 9)

### 2. The voice script
Writes `make_voice.py` (the same file as `tools/make_voice.py` in the repo). Just run it.

In [ ]:
%%writefile make_voice.py
"""
Tuldok voice maker: records every line the kid app says, with feeling, using an open voice model.
Run it in Google Colab with a GPU (Runtime > Change runtime type > T4 GPU).

Easiest: open tools/tuldok_voice.ipynb in Colab (it contains this script and all the steps).
By hand:
  Cell 1:  !pip install -q chatterbox-tts torchvision==0.21.0
           import os; os.kill(os.getpid(), 9)   # restarts Colab so the new packages load ("session crashed" is expected)
  Cell 2:  !wget -q -O make_voice.py https://raw.githubusercontent.com/akosidave31/tuldok/main/tools/make_voice.py
           %run -i make_voice.py --sample      # 6 test lines, plays them right here
  Cell 3:  %run -i make_voice.py               # all lines -> /content/tuldok-voice.zip (downloads it)
  (-i keeps the loaded model between runs, so later runs start faster)

Options:
  --exag 0.9          feeling: 0.5 plain, 0.9 happy (default), 1.3 very excited (too high can get shaky)
  --cfg 0.35          pace: lower = slower and more careful, 0.5 = model default
  --voice x.wav       copy a real person's voice from a 10-20 second clean WAV (only with that person's permission)
  --only "text"       remake only the lines containing this text, with a new take, e.g. --only "letter A"
  --fix "OLD=>NEW"    change how a word is read, e.g. --fix "letter A!=>letter Ay!" (repeatable)
  --engine kokoro     use Kokoro-82M instead (calmer, no feeling control). First run:
                      !pip install -q kokoro soundfile  and  !apt-get -qq install espeak-ng
  --kvoice af_heart   Kokoro voice name

  --out DIR           where clips are kept (default /content/tuldok-voice; the notebook can use Google Drive)

Running it again continues where it stopped (finished clips are kept).
Output: DIR.zip (default /content/tuldok-voice.zip) holding voice/*.mp3 and voice/manifest.js.
Unzip it into app/src/main/assets/ of the tuldok repo.

The line list below must match what assets/index.html says. A line without a clip
falls back to the tablet voice, and the grown-ups menu shows how many lines lack a clip.
"""
import argparse, contextlib, datetime, hashlib, io, json, os, random, re, subprocess, time, zipfile

import numpy as np
from scipy.io import wavfile

ap = argparse.ArgumentParser()
ap.add_argument('--sample', action='store_true')
ap.add_argument('--engine', default='chatterbox', choices=['chatterbox', 'kokoro'])
ap.add_argument('--exag', type=float, default=0.9)
ap.add_argument('--cfg', type=float, default=0.35)
ap.add_argument('--voice', default='')
ap.add_argument('--kvoice', default='af_heart')
ap.add_argument('--only', default='')
ap.add_argument('--fix', action='append', default=[])
ap.add_argument('--seed', type=int, default=7)
ap.add_argument('--out', default='/content/tuldok-voice')
args, _ = ap.parse_known_args()

# ---------------- every line the app says (keep in step with assets/index.html) ----------------
NUM = ['zero', 'one', 'two', 'three', 'four', 'five', 'six', 'seven', 'eight', 'nine', 'ten', 'eleven', 'twelve',
       'thirteen', 'fourteen', 'fifteen', 'sixteen', 'seventeen', 'eighteen', 'nineteen', 'twenty']
ALPHA = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'
CUE = dict(A='apple', B='ball', C='cat', D='dog', E='egg', F='fish', G='grapes', H='house', I='ice cream',
           J='juice', K='kite', L='lion', M='moon', N='nose', O='octopus', P='pig', Q='queen', R='rainbow',
           S='sun', T='tree', U='umbrella', V='violin', W='watermelon', X='box', Y='yo-yo', Z='zebra')
THINGS = ['apple', 'star', 'fish', 'flower', 'car', 'balloon', 'cat', 'dog', 'butterfly', 'mango', 'banana',
          'ball', 'bee', 'moon', 'frog', 'duck', 'watermelon', 'turtle', 'strawberry', 'bus']
LINE_SAYS = ['Help the bee fly to the flower!', 'Make rain! Draw from the top down!',
             'Draw round circles, like a ball!', 'Draw a line down, then a line across!',
             'Draw slanting lines, like a kite string!', 'Help the frog hop to the leaf!',
             'Swim like a fish to the shell!', 'Hop like a bunny to the carrot!', 'Loop all the way to the flower!']
SHAPES = ['circle', 'square', 'triangle', 'rectangle', 'star', 'oval', 'heart', 'diamond']
PRAISE = ['Yay! Great job!', 'Woohoo! You did it!', 'Wow! Awesome!', 'Super duper!', 'Hooray! You are a star!']
SAMPLE = ["Hi there! Let's play!", "Let's trace big letter A! A is for apple!", 'Yay! Great job!',
          'How many butterflies did you count?', 'Three!', "Oops! Let's count again!"]


def cap(s):
    return s[:1].upper() + s[1:]


def plural(w):
    special = {'fish': 'fish', 'mango': 'mangoes', 'bus': 'buses'}
    if w in special:
        return special[w]
    if re.search(r'[^aeiou]y$', w):
        return w[:-1] + 'ies'
    return w + 's'


def all_lines():
    out = ["Hi there! Let's play!", 'Start at the green dot!']
    for c in ALPHA:
        out.append(f"Let's trace big letter {c}! {c} is for {CUE[c]}!")
        out.append(f'Now the small letter {c}!')
    for n in range(0, 21):
        out.append(f"Let's trace the number {NUM[n]}!")
    out += LINE_SAYS
    for s in SHAPES:
        out.append(f"Let's trace {'an' if s[0] in 'aeiou' else 'a'} {s}!")
    for n in range(1, 21):
        out += [cap(NUM[n]) + '!', 'Yes! ' + cap(NUM[n]) + '!']
    for t in THINGS:
        out += [f"Let's count! Tap each {t}!", f'How many {plural(t)} did you count?', cap(t) + '!', 'Yes! ' + cap(t) + '!']
    out += ["Oops! Let's count again!", 'Draw a line to the same picture!', 'Match the big letter to the small letter!',
            'Match the number to the dots!', 'What comes next?', 'Hmm, look again!']
    out += PRAISE
    seen, uniq = set(), []
    for t in out:
        if key(t) not in seen:
            seen.add(key(t))
            uniq.append(t)
    return uniq


def key(t):  # same rule as vkey() in the app: lowercase, punctuation becomes one space
    return re.sub(r'[^a-z0-9]+', ' ', t.lower()).strip()


def fname(k):
    return re.sub(r' +', '-', k)[:40].strip('-') + '-' + hashlib.md5(k.encode()).hexdigest()[:6] + '.mp3'


FIXES = [f.split('=>', 1) for f in args.fix if '=>' in f]


def spoken(text):
    for old, new in FIXES:
        text = text.replace(old, new)
    return text


# ---------------- voice engines ----------------
def load_engine():
    import torch
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    if dev == 'cpu':
        print('!! No GPU. Use Runtime > Change runtime type > T4 GPU, or this will be very slow.', flush=True)
    # With "%run -i" this lives in the notebook, so the model loads once per session.
    cache = globals().setdefault('_tuldok_engines', {})
    if args.engine == 'kokoro':
        ck = ('kokoro', dev)
        if ck not in cache:
            from kokoro import KPipeline
            cache[ck] = KPipeline(lang_code='a', device=dev)
        pipe = cache[ck]

        def gen(text, seed):
            parts = [a for _, _, a in pipe(text, voice=args.kvoice, speed=0.95)]
            parts = [p.detach().cpu().numpy() if hasattr(p, 'detach') else np.asarray(p) for p in parts]
            return np.concatenate(parts).astype(np.float32), 24000
        return gen, f'Kokoro-82M ({args.kvoice}, Apache 2.0)'

    ck = ('chatterbox', dev, args.voice)  # a copied voice changes the model's state, so it gets its own load
    if ck not in cache:
        cache.clear()
        if dev == 'cuda':
            torch.cuda.empty_cache()
        from chatterbox.tts import ChatterboxTTS
        print('Loading Chatterbox (first time downloads about 2-3 GB)...', flush=True)
        cache[ck] = ChatterboxTTS.from_pretrained(device=dev)
    model = cache[ck]

    def gen(text, seed):
        torch.manual_seed(seed)
        random.seed(seed)
        np.random.seed(seed % (2 ** 32))
        kw = dict(exaggeration=args.exag, cfg_weight=args.cfg)
        if args.voice:
            kw['audio_prompt_path'] = args.voice
        with torch.inference_mode(), contextlib.redirect_stderr(io.StringIO()):
            wav = model.generate(text, **kw)
        return wav.squeeze().detach().cpu().numpy().astype(np.float32), model.sr
    return gen, f'Chatterbox by Resemble AI (MIT), exaggeration {args.exag}, cfg {args.cfg}' + (', cloned voice' if args.voice else '')


# ---------------- clean-up: trim silence, even loudness, soft edges ----------------
def tidy(x, sr):
    x = np.nan_to_num(np.asarray(x, dtype=np.float32))
    if x.size < sr // 10:
        return None
    win = max(1, int(sr * 0.02))
    env = np.convolve(np.abs(x), np.ones(win) / win, mode='same')
    thr = max(env.max() * 0.05, 1e-4)
    on = np.where(env > thr)[0]
    if on.size == 0:
        return None
    s, e = max(0, on[0] - int(sr * 0.04)), min(x.size, on[-1] + int(sr * 0.15))
    y = x[s:e].copy()
    f = min(int(sr * 0.012), y.size // 4)
    if f > 0:
        ramp = np.linspace(0, 1, f, dtype=np.float32)
        y[:f] *= ramp
        y[-f:] *= ramp[::-1]
    voiced = y[env[s:e] > thr]
    rms = float(np.sqrt(np.mean(voiced ** 2))) if voiced.size else float(np.sqrt(np.mean(y ** 2)))
    g = min(10 ** (-17 / 20) / max(rms, 1e-6), 0.95 / max(float(np.abs(y).max()), 1e-6))
    return y * g


def length_ok(text, secs):
    w = max(1, len(re.findall(r"[A-Za-z0-9']+", text)))
    lo, hi = 0.2 * w, 0.6 * w + 1.4
    return lo <= secs <= hi, (lo + hi) / 2


def make_clip(gen, text, seed):
    best = None
    for take in range(4):  # a take that is far too long/short (babbling, cut off) is redone
        x, sr = gen(spoken(text), seed + 101 * take)
        y = tidy(x, sr)
        if y is None:
            continue
        ok, mid = length_ok(text, y.size / sr)
        if ok:
            return y, sr, take
        if best is None or abs(y.size / sr - mid) < abs(best[0].size / best[1] - mid):
            best = (y, sr)
    if best is None:
        raise RuntimeError('no audio for: ' + text)
    return best[0], best[1], 9


def save_mp3(y, sr, path):
    tmp = path[:-4] + '.wav'
    wavfile.write(tmp, sr, np.clip(y * 32767, -32768, 32767).astype(np.int16))
    subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', tmp, '-ac', '1', '-codec:a', 'libmp3lame',
                    '-b:a', '64k', path], check=True)
    os.remove(tmp)


# ---------------- run ----------------
def main():
    full = all_lines()
    lines = SAMPLE if args.sample else full
    root = '/content/tuldok-sample' if args.sample else args.out
    vdir = os.path.join(root, 'voice')
    os.makedirs(vdir, exist_ok=True)
    only = args.only.lower()
    todo = []
    for t in lines:
        p = os.path.join(vdir, fname(key(t)))
        if only:
            if only in t.lower():
                todo.append(t)
        elif args.sample or not os.path.exists(p):
            todo.append(t)
    print(f'{len(full)} lines in the app, {len(todo)} to make now.', flush=True)
    if only and not todo:
        print(f'No line contains "{args.only}".')
    gen, label = (load_engine() if todo else (None, 'unchanged'))
    t0 = time.time()
    for i, text in enumerate(todo, 1):
        k = key(text)
        seed = args.seed + int(hashlib.md5(k.encode()).hexdigest()[:6], 16) % 100000
        if only:
            seed += random.randint(1, 10 ** 6)  # a fresh take
        t1 = time.time()
        y, sr, take = make_clip(gen, text, seed)
        save_mp3(y, sr, os.path.join(vdir, fname(k)))
        left = (time.time() - t0) / i * (len(todo) - i)
        note = '' if take == 0 else (' (retake)' if take < 9 else ' (CHECK: odd length)')
        print(f'[{i:3d}/{len(todo)}] {time.time() - t1:4.1f}s  {y.size / sr:3.1f}s audio  ~{left / 60:4.1f} min left  {text}{note}', flush=True)

    if args.sample:
        from IPython.display import Audio, display
        for t in SAMPLE:
            print(t)
            display(Audio(os.path.join(vdir, fname(key(t)))))
        print('Like it? Run the full set:  %run make_voice.py' + (f' --exag {args.exag}' if args.exag != 0.9 else ''))
        return

    clips = {key(t): fname(key(t)) for t in full if os.path.exists(os.path.join(vdir, fname(key(t))))}
    meta_path = os.path.join(vdir, 'settings.json')
    if todo and not only:
        json.dump({'model': label}, open(meta_path, 'w'))
    model_label = json.load(open(meta_path)).get('model', label) if os.path.exists(meta_path) else label
    manifest = {'model': model_label, 'made': datetime.date.today().isoformat(), 'clips': clips}
    with open(os.path.join(vdir, 'manifest.js'), 'w') as fh:
        fh.write('/* Made by tools/make_voice.py. Maps each line (lowercase words) to its clip. */\n')
        fh.write('window.TULDOK_VOICE = ' + json.dumps(manifest, indent=1) + ';\n')
    with open(os.path.join(vdir, 'CREDITS.txt'), 'w') as fh:
        fh.write('Voice clips generated with ' + model_label + '.\n')
        if 'Chatterbox' in model_label:
            fh.write("Chatterbox: https://github.com/resemble-ai/chatterbox (MIT License). Clips carry Resemble AI's inaudible Perth watermark.\n")
        else:
            fh.write('Kokoro-82M: https://huggingface.co/hexgrad/Kokoro-82M (Apache 2.0).\n')
    zpath = args.out.rstrip('/') + '.zip'
    with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
        keep = set(clips.values()) | {'manifest.js', 'CREDITS.txt'}
        for f in sorted(os.listdir(vdir)):
            if f in keep:
                z.write(os.path.join(vdir, f), 'voice/' + f)
    missing = [t for t in full if key(t) not in clips]
    size = os.path.getsize(zpath) / 1e6
    print(f'\nDone: {len(clips)}/{len(full)} clips, {size:.1f} MB -> {zpath}', flush=True)
    if missing:
        print('Still missing (run again to finish):', missing[:5])
    try:
        from google.colab import files
        files.download(zpath)
    except Exception:
        print('Download ' + zpath + ' from the Files panel on the left (tap ⋮ > Download), or from Google Drive if it is saved there.')


main()

### 3. Optional: copy a real person's voice
Skip this to use Chatterbox's own voice. To copy a voice, record **10–20 seconds** of clear, happy speech in a quiet room
(WAV is best) from someone who **agrees** to it, upload it here, then type its file name in `VOICE_WAV` in cell 4.

In [ ]:
#@title Upload a voice file (optional)
from google.colab import files
up = files.upload()
print('Uploaded:', list(up), '-> type this name in VOICE_WAV in cell 4')

In [ ]:
#@title 4. Settings
#@markdown **Feeling** (exaggeration): 0.5 plain · 0.9 happy · 1.2+ very excited (too high can get shaky)
EXAG = 0.9 #@param {type:"slider", min:0.3, max:1.5, step:0.05}
#@markdown **Pace** (cfg): lower = slower and more careful · 0.5 = model default
CFG = 0.35 #@param {type:"slider", min:0.2, max:0.8, step:0.05}
#@markdown **Copied voice**: file name from step 3, or leave empty
VOICE_WAV = "" #@param {type:"string"}
#@markdown **Engine**: kokoro is calmer, with no feeling control (needs the Kokoro cell at the bottom first)
ENGINE = "chatterbox" #@param ["chatterbox", "kokoro"]
KOKORO_VOICE = "af_heart" #@param {type:"string"}
#@markdown **Save to Drive**: keeps finished clips if Colab disconnects (asks for Google Drive access once)
SAVE_TO_DRIVE = True #@param {type:"boolean"}
import os
OUT = '/content/tuldok-voice'
if SAVE_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        OUT = '/content/drive/MyDrive/tuldok-voice'
    except Exception as e:
        print('Drive not mounted, saving in Colab only:', e)
OPTS = f'--exag {EXAG} --cfg {CFG} --engine {ENGINE} --kvoice {KOKORO_VOICE} --out "{OUT}"'
if VOICE_WAV.strip():
    if os.path.exists(VOICE_WAV.strip()):
        OPTS += f' --voice "{VOICE_WAV.strip()}"'
    else:
        print('!! VOICE_WAV not found:', VOICE_WAV, '(using the built-in voice)')
print('Settings:', OPTS)

In [ ]:
#@title 5. Listen to 6 test lines (first run downloads the model, 2-3 GB)
%run -i make_voice.py --sample {OPTS}

In [ ]:
#@title 6. Make all lines → tuldok-voice.zip (about 10-25 min on a T4; run again to continue after a disconnect)
%run -i make_voice.py {OPTS}

### 7. Optional: remake lines that sound wrong
`ONLY` picks every line containing that text (e.g. `letter A`, `butterflies`, `Super duper`) and records a fresh take.
`FIX` changes how something is read, as `OLD=>NEW`, e.g. `letter A! A is=>letter Ay! Ay is`.
It then remakes the zip and downloads it again.

In [ ]:
#@title Remake lines
ONLY = "letter A" #@param {type:"string"}
FIX = "" #@param {type:"string"}
EXTRA = f' --fix "{FIX}"' if '=>' in FIX else ''
%run -i make_voice.py --only "{ONLY}" {EXTRA} {OPTS}

In [ ]:
#@title Optional: install Kokoro (only if you set ENGINE = kokoro)
get_ipython().system('pip install -q kokoro soundfile')
get_ipython().system('apt-get -qq -y install espeak-ng > /dev/null')
print('Kokoro ready. Set ENGINE to kokoro in cell 4, run cell 4, then cell 5.')

### Put the voice into the app (Termux)
If the download didn't start, get `tuldok-voice.zip` from **Google Drive › My Drive** (or the Files panel on the left)
into your phone's Downloads. Then:

```
rm -rf ~/tuldok-app/app/src/main/assets/voice; unzip -oq "$(ls -t ~/storage/downloads/tuldok-voice*.zip | head -1)" -d ~/tuldok-app/app/src/main/assets/; cd ~/tuldok-app; git add -A; git commit -m "Add Chatterbox voice clips"; git push; bash ~/tuldok-getapk.sh
```

In the app, hold ⚙️ to open the grown-ups menu: it shows how many voice clips are installed, and *Voice style → Happy teacher* plays them.